# Kontinuitet u suženju cijevi

**Poglavlje U08: Kontrolni volumen i kontinuitet**

Ovaj interaktivni prikaz nadopunjuje jednadžbu kontinuiteta za nestlačivi fluid u cijevi sa suženjem. Mijenjanjem promjera ulaza i izlaza te volumenskog protoka prati se promjena brzine strujanja duž cijevi.

## Cilj

U cijevi s promjenjivim presjekom volumenski protok ostaje konstantan duž cijevi, dok se brzina mijenja obrnuto proporcionalno površini presjeka. Prikaz omogućuje:

1. mijenjanje promjera ulaza $D_1$;
2. mijenjanje promjera izlaza $D_2$;
3. mijenjanje protoka $Q$;
4. praćenje brzina $v_1$ i $v_2$ te promjene brzine duž cijevi.

## Pretpostavke modela

- nestlačivi fluid (voda);
- stacionarno strujanje;
- jednodimenzijski profil brzina u svakom presjeku;
- bez gubitaka po cijeloj cijevi.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

## Računski model

Iz jednadžbe kontinuiteta za nestlačivi fluid:

$$Q = A_1 v_1 = A_2 v_2,$$

iz čega slijedi izlazna brzina:

$$v_2 = v_1\,\frac{A_1}{A_2} = v_1\left(\frac{D_1}{D_2}\right)^2.$$

Brzina raste tamo gdje se cijev sužava i pada tamo gdje se širi. Jednadžba ne ovisi o tlaku ni o energiji — to su posebne teme narednih poglavlja.

In [ ]:
def kontinuitet(D1_mm, D2_mm, Q_Lpsek):
    D1 = D1_mm / 1000.0
    D2 = D2_mm / 1000.0
    Q = Q_Lpsek / 1000.0  # m^3/s
    A1 = np.pi * D1**2 / 4
    A2 = np.pi * D2**2 / 4
    v1 = Q / A1
    v2 = Q / A2
    return {'v1': v1, 'v2': v2, 'A1': A1, 'A2': A2}

## Interaktivni prikaz

Klizačima u nastavku biraju se promjeri ulaza i izlaza te ukupni protok. Gornji prikaz pokazuje presjek cijevi, donji pokazuje brzinu duž osi cijevi.

In [ ]:
def kontinuitet_prikaz(D1_mm, D2_mm, Q_Lpsek):
    r = kontinuitet(D1_mm, D2_mm, Q_Lpsek)

    fig, (ax_geo, ax_v) = plt.subplots(
        2, 1, figsize=(9, 6.5),
        gridspec_kw={'height_ratios': [1, 1.4]}
    )

    # Geometrija cijevi (suženje 0.3 do 0.5 m)
    x = np.linspace(0, 1.0, 400)
    D = np.full_like(x, D1_mm, dtype=float)
    maska = (x >= 0.3) & (x < 0.5)
    D[maska] = D1_mm + (D2_mm - D1_mm) * (x[maska] - 0.3) / 0.2
    D[x >= 0.5] = D2_mm
    A = np.pi * (D / 1000)**2 / 4
    v = (Q_Lpsek / 1000) / A

    # Cijev (presjek)
    ax_geo.fill_between(x, -D/2, D/2, color='#aed6f1', alpha=0.7)
    ax_geo.plot(x, D/2, color='#1565c0', lw=2)
    ax_geo.plot(x, -D/2, color='#1565c0', lw=2)
    ax_geo.set_xlim(0, 1.0)
    ax_geo.set_ylim(-max(D1_mm, D2_mm)*0.7,
                     max(D1_mm, D2_mm)*0.7)
    ax_geo.set_ylabel('polumjer (mm)')
    ax_geo.set_xticks([])
    ax_geo.set_title(
        f'$D_1$ = {D1_mm:.0f} mm,  $D_2$ = {D2_mm:.0f} mm,  '
        f'$Q$ = {Q_Lpsek:.1f} L/s'
    )

    # Brzina
    ax_v.plot(x, v, color='#c62828', lw=2.2)
    ax_v.fill_between(x, 0, v, color='#c62828', alpha=0.2)
    ax_v.set_xlabel('osna koordinata (m)')
    ax_v.set_ylabel('brzina (m/s)')
    ax_v.grid(ls=':', alpha=0.5)
    ax_v.axhline(r['v1'], color='gray', ls=':', lw=0.8)
    ax_v.text(0.02, r['v1'], f'$v_1$ = {r["v1"]:.2f} m/s',
              color='gray', fontsize=9, va='bottom')
    ax_v.axhline(r['v2'], color='gray', ls=':', lw=0.8)
    ax_v.text(0.98, r['v2'], f'$v_2$ = {r["v2"]:.2f} m/s',
              color='gray', fontsize=9, va='bottom', ha='right')

    plt.tight_layout()
    plt.show()


interact(
    kontinuitet_prikaz,
    D1_mm=FloatSlider(min=30, max=200, step=5, value=100,
                       description='$D_1$ (mm)',
                       layout=Layout(width='420px')),
    D2_mm=FloatSlider(min=10, max=150, step=5, value=40,
                       description='$D_2$ (mm)',
                       layout=Layout(width='420px')),
    Q_Lpsek=FloatSlider(min=0.5, max=30, step=0.5, value=8,
                         description='$Q$ (L/s)',
                         layout=Layout(width='420px'))
);

## Pitanja za istraživanje

1. **Skala s omjerom presjeka.** Za $D_2 = D_1 / 2$, koliko je puta veća izlazna brzina od ulazne? A za $D_2 = D_1 / 4$? Što je vidljivo iz formule i grafa?

2. **Granični slučaj jednakog presjeka.** Pri $D_1 = D_2$, kakva je raspodjela brzine duž cijevi? Vrijedi li tada još uvijek jednadžba kontinuiteta?

3. **Energetske posljedice.** Iako jednadžba kontinuiteta ne spominje tlak, povećanje brzine u suženju u idealnom fluidu neminovno znači pad tlaka. Što kaže Bernoullijeva jednadžba o tome (poglavlje U09)?

4. **Stvarni profil brzina.** Zašto u stvarnoj cijevi profil brzina nije jednolik nego približno parabolan (laminarno) ili polako-jednolik (turbulentno)? Što to znači za točnost predviđanja $v_2$ iz ovog modela?

## Veza s teorijom poglavlja

Ovaj prikaz materijalizira jednadžbu kontinuiteta iz poglavlja U08 — najjednostavniji oblik bilance mase za nestlačivi fluid u jednoj cijevi. Ista logika proširuje se na čvorove (zbroj ulaznih protoka jednak zbroju izlaznih) i na spremnike s akumulacijom (razlika ulaza i izlaza jednaka brzini promjene volumena). U poglavlju U09 se na ovu osnovu dodaje energetska bilanca preko Bernoullija.